# Phase 1：文档解析与中文分块

## 今天交付什么？

今天不是学习“什么是 RAG”然后结束，而是把真实文件加工成后续检索可以直接使用的 `data/processed/chunks.json`。这一步是整个项目的地基：如果来源、页码和文本边界丢失，后面的检索即使命中了，也无法向用户证明“答案来自哪里”。

**完成后你要能回答：**

1. 为什么不能把整本 PDF 直接塞给检索器？
2. 为什么 Chunk 需要 `id/source/page/text`，而不是只保存一段字符串？
3. `chunk_size` 和 `overlap` 改变时，信息完整性、索引成本和重复召回如何变化？

**最终产物：** 一份稳定、可追溯、可重复生成的 Chunk 数据集。今天的每个实验都必须服务于这个产物。

## Evidence Quest 任务卡：Phase 1 总览：建立证据地基

**你的身份：** 文档考古组组长  
**案件背景：** 先把散落文件整理成每一条都能回到原文的证据单元，后面的搜索比赛才有可信的赛道。

### 本关专业 Goal

完成从 PDF/Markdown 到稳定 chunks.json 的第一条可复现流水线。

### 你要交付的作品

**文档考古工具总览 + chunks.json**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：证据地基组长  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 0. 前置知识：只补今天会用到的

你不需要先学完机器学习。够用的知识是：

| 知识 | 今天怎么用 | 验收方式 |
| --- | --- | --- |
| `Path` 和文件编码 | 批量找到 `.md/.txt/.pdf` 并读取 UTF-8 | 能指出输入目录和输出文件 |
| 列表、字典、函数 | 表示文档和 Chunk | 能读取一条 Chunk 的字段 |
| 字符串长度和切片 | 限制 Chunk 大小、保留重叠上下文 | 能解释 `text[a:b]` |
| 元数据 | 保留 source/page/headings | 能从检索结果回到原文件 |

**学习原则：** 看到一个知识点后马上在项目数据上做一个小实验；如果它没有改变代码、数据或判断，就先不扩展。

In [1]:
from pathlib import Path
import json
import sys


def find_project_root() -> Path:
    """兼容从项目根目录、notebooks 目录或 JupyterLab 启动目录运行。"""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase1'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase1
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


## 1. 先看全链路：为什么要分块？

一个 RAG 系统通常经历：

```text
原始文件 -> 解析成文本和元数据 -> 分块 -> 建索引 -> 查询 -> 返回引用
```

分块解决的是一个具体矛盾：

- Chunk 太大：一次召回带来很多无关内容，排序不精确，也更占上下文窗口。
- Chunk 太小：关键词或事实可能被切到两个 Chunk 中，单个 Chunk 失去完整语义。
- 有 overlap：相邻 Chunk 共享边界文字，能降低“事实刚好被切开”的概率；代价是 Chunk 数、索引体积和重复结果增加。

先不要背结论。下面从项目输入开始观察。

In [3]:
from phase1_doc_parser.parser import parse_file

input_dir = ROOT / "phase1_doc_parser" / "examples" / "input"
files = sorted(path for path in input_dir.iterdir() if path.is_file())
print("输入目录:", input_dir)
print("文件:", [path.name for path in files])

for path in files:
    documents = parse_file(path)
    print(f"\n{path.name}: 解析为 {len(documents)} 个文档片段")
    for document in documents:
        print({"source": document.source, "page": document.page, "chars": len(document.text), "metadata": document.metadata})

输入目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input
文件: ['quickstart.md', 'retrieval-notes.md']

quickstart.md: 解析为 1 个文档片段
{'source': 'D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md', 'page': None, 'chars': 138, 'metadata': {'format': 'markdown', 'headings': ['Phase 1 Quickstart', 'Chunk 策略']}}

retrieval-notes.md: 解析为 1 个文档片段
{'source': 'D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\retrieval-notes.md', 'page': None, 'chars': 145, 'metadata': {'format': 'markdown', 'headings': ['Retrieval Notes']}}


### 你刚才观察到的是什么？

`parse_file` 没有急着把所有东西变成 Chunk。它先统一输出 `ParsedDocument`：

- Markdown/TXT 通常是一份文档，`page=None`。
- PDF 按页输出，`page=1,2,...`，这样一个 Chunk 才能精确引用页码。
- Markdown heading 被放入 metadata，而不是混进一个无法查询的隐藏状态。

这叫**数据契约**：后续模块只依赖稳定字段，不需要知道文件是怎么解析的。换句话说，Phase 2 不应该重新猜“这段文字来自哪个文件”。

In [4]:
document = parse_file(files[0])[0]
print("文本前 240 个字符:\n", document.text[:240])
print("\n字段含义:")
print("text = 可检索正文")
print("source = 用户需要回看的原始文件")
print("page = PDF 页码；Markdown 没有页码时为 None")
print("metadata = 不参与正文检索但有助于解释的附加信息")

文本前 240 个字符:
 # Phase 1 Quickstart

文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。

## Chunk 策略

先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。

字段含义:
text = 可检索正文
source = 用户需要回看的原始文件
page = PDF 页码；Markdown 没有页码时为 None
metadata = 不参与正文检索但有助于解释的附加信息


## 2. Recursive Splitter：它到底在递归什么？

本项目的分隔符顺序是：

```text
段落(\n\n) -> 换行(\n) -> 中文句号/问号/分号 -> 逗号 -> 空格 -> 单字符
```

“递归”不是神秘算法：先尝试用最能保留语义的边界切；如果某一段仍然超过上限，就降级到更细的边界；最后实在没有边界，才按字符切。这样做的原因是**优先保留自然语言结构，同时保证硬性长度上限**。

长度上限不是为了让数字好看，而是为了控制一次检索返回的上下文大小。当前实现用 Python 的 `len` 统计字符数，适合教学和中文 baseline；生产系统还应记录 tokenizer token 数，因为模型上下文窗口按 token 计费和限制。

In [5]:
from phase1_doc_parser.splitter import RecursiveSplitter

sample = "第一段：检索需要来源。\n\n第二段：overlap 保留跨边界上下文。\n第三段：如果段落过长，再按标点继续切。"
splitter = RecursiveSplitter(chunk_size=36, overlap=8)
chunks = splitter.split(sample)
for index, chunk in enumerate(chunks):
    print(f"Chunk {index} ({len(chunk)} chars): {chunk!r}")

Chunk 0 (34 chars): '第一段：检索需要来源。\n\n第二段：overlap 保留跨边界上下文。'
Chunk 1 (27 chars): '跨边界上下文。\n第三段：如果段落过长，再按标点继续切。'


### 用一个反例理解 overlap

假设关键事实是：`答案在句子末尾，证据在下一段开头`。如果恰好在边界切开，检索其中一半时，用户看到的文本可能无法独立解释问题。overlap 会把前一个 Chunk 的尾部复制到下一个 Chunk 的头部，增加两边同时含有线索的机会。

但 overlap 不是越大越好：如果 `overlap=chunk_size-1`，相邻 Chunk 几乎重复，索引变大，top-k 可能被同一段内容占满，反而降低结果多样性。下面只改变 overlap，其他变量保持不变。

In [6]:
def summarize_split(chunk_size: int, overlap: int) -> dict[str, float | int]:
    result = RecursiveSplitter(chunk_size=chunk_size, overlap=overlap).split(document.text)
    lengths = [len(item) for item in result]
    return {
        "chunk_size": chunk_size,
        "overlap": overlap,
        "count": len(result),
        "avg_chars": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "max_chars": max(lengths, default=0),
    }

for overlap in (0, 16, 32):
    print(summarize_split(64, overlap))

{'chunk_size': 64, 'overlap': 0, 'count': 3, 'avg_chars': 44.7, 'max_chars': 59}
{'chunk_size': 64, 'overlap': 16, 'count': 3, 'avg_chars': 48.7, 'max_chars': 64}
{'chunk_size': 64, 'overlap': 32, 'count': 3, 'avg_chars': 48.7, 'max_chars': 64}


**如何读这个实验：**

- `count` 增大说明重复上下文带来了更多索引项。
- `max_chars` 不应超过 `chunk_size`，这是硬约束。
- `avg_chars` 只能描述数据形状，不能证明检索质量变好；质量要在 Phase 2 用 qrels 验证。

这一区分很重要：**分块统计是原因线索，Recall/MRR 才是检索证据。**

In [7]:
from phase1_doc_parser.main import build_chunks

splitter = RecursiveSplitter(chunk_size=128, overlap=32)
project_chunks = build_chunks(input_dir, splitter)
print("生成 Chunk 数:", len(project_chunks))
print(json.dumps(project_chunks[0], ensure_ascii=False, indent=2))

required = {"id", "text", "source", "page", "chunk_index", "metadata"}
assert project_chunks and required <= project_chunks[0].keys()
assert all(item["text"] and len(item["text"]) <= 128 for item in project_chunks)
assert all(item["id"] and item["source"] for item in project_chunks)
print("数据契约检查通过：正文、来源、ID 和长度约束都满足。")

生成 Chunk 数: 4
{
  "id": "3692b05e025373a8",
  "text": "# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略",
  "source": "D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md",
  "page": null,
  "chunk_index": 0,
  "metadata": {
    "format": "markdown",
    "headings": [
      "Phase 1 Quickstart",
      "Chunk 策略"
    ]
  }
}
数据契约检查通过：正文、来源、ID 和长度约束都满足。


## 3. 可重复性：为什么 ID 不能每次随机？

评估时我们要知道“这次排名变了，是算法变了，还是文档 ID 变了”。因此项目使用由 `source/page/index/text` 计算出来的稳定哈希作为 Chunk ID：同一输入和同一参数会得到同一 ID；正文或边界改变时，ID 会变化，提醒我们索引需要更新。

In [8]:
run_a = build_chunks(input_dir, RecursiveSplitter(chunk_size=128, overlap=32))
run_b = build_chunks(input_dir, RecursiveSplitter(chunk_size=128, overlap=32))
assert [item["id"] for item in run_a] == [item["id"] for item in run_b]
print("两次运行 ID 完全一致，共", len(run_a), "个 Chunk")

output_path = ROOT / "data" / "processed" / "chunks.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(run_a, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写入:", output_path)

两次运行 ID 完全一致，共 4 个 Chunk
已写入: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\chunks.json


## 4. 故意制造错误：理解边界比记住 API 更重要

好的工程学习不只验证正常输入，也要观察系统如何拒绝危险参数。`overlap` 必须小于 `chunk_size`，否则“保留上下文”会覆盖整个 Chunk，算法无法前进；不支持的文件格式也必须尽早报错，而不是静默生成空数据。

In [9]:
try:
    RecursiveSplitter(chunk_size=10, overlap=10)
except ValueError as exc:
    print("非法参数被拒绝:", exc)

try:
    parse_file(input_dir / "not-supported.csv")
except (ValueError, FileNotFoundError) as exc:
    print("不支持的输入会显式失败:", exc)

非法参数被拒绝: overlap must be in [0, chunk_size)
不支持的输入会显式失败: Unsupported file type: .csv


## Phase 1 阶段闸门

完成下面清单后才进入 Phase 2：

- [ ] 能解释解析、分块、索引三者的边界。
- [ ] 能从任意 Chunk 找到 `source/page`，并说明为什么保存它们。
- [ ] 至少比较 3 组 `chunk_size/overlap`，但不把 Chunk 数当作质量指标。
- [ ] 输出 `data/processed/chunks.json`，且重复运行 ID 稳定。
- [ ] 能解释 overlap 的收益、索引成本和重复召回风险。

**项目交付物：** `chunks.json` + 一段分块策略结论。下一阶段只读取这份真实产物，建立 BM25 baseline。

## Boss Challenge：从输入目录完整跑一次解析和分块，并解释一个 Chunk 如何回到原文。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [10]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [11]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/chunks.json', 'phase1_doc_parser/output/chunks.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\chunks.json', 'phase1_doc_parser\\output\\chunks.json']
